In [ ]:
# Step 1: Install the required libraries for the project.
%pip install pandas numpy scikit-learn

In [ ]:
# Step 2: Import all libraries needed for data handling and similarity calculations.
import pandas as pd
import numpy as np
import ast
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Step 3: Load the two TMDB datasets and merge them using the movie title.
movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")

movies = movies.merge(credits, on="title")
movies.head()

In [ ]:
# Step 4: Keep only the columns needed for building the recommendation system.
movies = movies[["movie_id", "title", "overview", "genres", "keywords", "cast", "crew"]]
movies.head()

In [ ]:
# Step 5: Remove rows that contain missing values.
movies.dropna(inplace=True)
movies.isnull().sum()

In [ ]:
# Step 6.1: Create a helper function to extract names from genres and keywords.
def convert(text):
    items = []
    for element in ast.literal_eval(text):
        items.append(element["name"])
    return items

In [ ]:
# Step 6.2: Create a helper function to extract the top 3 actors from the cast column.
def convert3(text):
    items = []
    counter = 0
    for element in ast.literal_eval(text):
        if counter != 3:
            items.append(element["name"])
            counter += 1
        else:
            break
    return items

In [ ]:
# Step 6.3: Create a helper function to extract only the director name from the crew column.
def fetch_director(text):
    items = []
    for element in ast.literal_eval(text):
        if element["job"] == "Director":
            items.append(element["name"])
            break
    return items

In [ ]:
# Step 6.4: Apply all preprocessing functions to the relevant columns.
movies["genres"] = movies["genres"].apply(convert)
movies["keywords"] = movies["keywords"].apply(convert)
movies["cast"] = movies["cast"].apply(convert3)
movies["crew"] = movies["crew"].apply(fetch_director)
movies.head()

In [ ]:
# Step 7: Split overview text into words and remove spaces inside multi-word names.
movies["overview"] = movies["overview"].apply(lambda x: x.split())

def collapse(items):
    return [item.replace(" ", "") for item in items]

movies["genres"] = movies["genres"].apply(collapse)
movies["keywords"] = movies["keywords"].apply(collapse)
movies["cast"] = movies["cast"].apply(collapse)
movies["crew"] = movies["crew"].apply(collapse)
movies.head()

In [ ]:
# Step 8: Combine overview, genres, keywords, cast, and crew into one tags column.
movies["tags"] = movies["overview"] + movies["genres"] + movies["keywords"] + movies["cast"] + movies["crew"]
movies["tags"] = movies["tags"].apply(lambda x: " ".join(x))
movies["tags"] = movies["tags"].apply(lambda x: x.lower())
movies[["title", "tags"]].head()

In [ ]:
# Step 9: Keep only the final columns required for recommendations.
movies = movies[["movie_id", "title", "tags"]].copy()
movies.head()

In [ ]:
# Step 10: Convert text data into numerical vectors using CountVectorizer.
cv = CountVectorizer(max_features=5000, stop_words="english")
vectors = cv.fit_transform(movies["tags"]).toarray()
vectors.shape

In [ ]:
# Step 11: Compute cosine similarity between all movie vectors.
similarity = cosine_similarity(vectors)
similarity.shape

In [ ]:
# Step 12: Create the recommendation function to return the top 5 similar movies.
def recommend(movie_name):
    movie_matches = movies[movies["title"] == movie_name]
    if movie_matches.empty:
        raise ValueError(f"Movie '{movie_name}' not found in the dataset.")

    movie_index = movie_matches.index[0]
    distances = list(enumerate(similarity[movie_index]))
    distances = sorted(distances, reverse=True, key=lambda x: x[1])

    recommendations = []
    for item in distances[1:6]:
        recommendations.append(movies.iloc[item[0]]["title"])

    return recommendations

In [ ]:
# Step 13: Test the recommendation function with the movie Avatar.
recommend("Avatar")

In [ ]:
# Step 14: Save the processed movies dataframe and similarity matrix as pickle files.
import pickle

with open("movies.pkl", "wb") as movie_file:
    pickle.dump(movies, movie_file)

with open("similarity.pkl", "wb") as similarity_file:
    pickle.dump(similarity, similarity_file)

print("movies.pkl and similarity.pkl saved successfully.")